### ChatBot Evaluation

In [1]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [2]:
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY", "")

In [6]:
## create the datapoints

from langsmith import Client
client = Client()

## define the dataset - test data
dataset_name = "Simple Chatbot Evaluation"
dataset = client.create_dataset(dataset_name)

examples = [
    {
        "inputs": {"question": "What is LangChain?"},
        "outputs": {"answer": "A framework for building LLM applications"},
    },
    {
        "inputs": {"question": "What is LangSmith?"},
        "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
    },
    {
        "inputs": {"question": "What is OpenAI?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    },
    {
        "inputs": {"question": "What is Google?"},
        "outputs": {"answer": "A technology company known for search"},
    },
    {
        "inputs": {"question": "What is Mistral?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    }
]

client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

{'example_ids': ['54ad3f84-c7d9-4781-931b-d5555ca21dc6',
  '61a215f6-936a-4c0b-a59f-4b8ef961a7f4',
  '10525ded-bc2b-4bfb-aad8-c664c554fa49',
  '6c7bcc9d-9725-40cb-9d58-f0c7f8e52e4e',
  'e3272439-d230-4133-84fc-209610dc5709'],
 'count': 5,
 'as_of': '2026-07-10T17:35:54.273307505Z'}

### Define Metrics (LLM as a Judge)

In [3]:
from google import genai
from langsmith import wrappers

gemini_client = wrappers.wrap_gemini(genai.Client(api_key=os.environ.get("GEMINI_API_KEY")))

C:\Users\agarw\AppData\Local\Temp\ipykernel_572\2379368207.py:4: LangSmithBetaWarning: Function wrap_gemini is in beta.
  gemini_client = wrappers.wrap_gemini(genai.Client(api_key=os.environ.get("GEMINI_API_KEY")))


In [4]:
# To list and print all available models
for model in gemini_client.models.list():
    print(model.name)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

In [5]:
eval_instructions = "You are an expert professor specialized in grading students' answers to questions"

In [6]:
from google.genai import types

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""
        Your are grading the following question:
        {inputs['question']}
        Here is the real answer:
        {reference_outputs['answer']}
        You are grading the following predicted answer:
        {outputs['response']}
        Respond with CORRECT or INCORRECT:
        Grade:
    """

    response = gemini_client.models.generate_content(
        model = "gemini-3.1-flash-lite",
        contents = user_content,
        config = types.GenerateContentConfig(
            system_instruction = eval_instructions,
            temperature = 0.0
        )
    )

    return response.text.strip() == "CORRECT"

In [7]:
## concisions: checks whether the actual output is less than 2x the length of the expected result

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

### Run the Evaluation

In [8]:
default_instructions = "Respond to the users questions in a short, concise manner (one short sentence)"

In [9]:
from google.genai import types

def my_app(question: str, model: str = "gemini-3.1-flash-lite", instructions: str = default_instructions) -> str:
    response = gemini_client.models.generate_content(
        model = model,
        contents = question,
        config = types.GenerateContentConfig(
            system_instruction = instructions,
            temperature = 0.0
        )
    )
    return response.text

In [10]:
### call my_app for every datapoints
def ls_target(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"])}

In [11]:
### test cell
print(ls_target({"question": "Hello"}))

c:\Users\agarw\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


{'response': 'Hello! How can I help you today?'}


In [12]:
### run our evaluation

experiment_results = client.evaluate(
    ls_target,
    data = dataset_name,
    evaluators = [correctness, concision],
    experiment_prefix="gemini-3.1-flash-lite-mini-chatbot"
)

NameError: name 'client' is not defined

### Evaluation for RAG

In [13]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap = 0
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

# Reduce the number of document splits to process to stay within API quota
# The free tier quota for 'embed_content_free_tier_requests' is 100.
# We'll use a small subset here, e.g., the first 20 splits.
doc_splits_subset = doc_splits[:20]

# Add the document chunks to the "vector store" using GoogleGenerativeAIEmbeddings
vectorStore = InMemoryVectorStore.from_documents(
    documents=doc_splits_subset,
    embedding=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"),
)

# With lanchain we can easily turn any vector store into a retrieval component:
retriever = vectorStore.as_retriever(k=6)

C:\Users\agarw\AppData\Local\Temp\ipykernel_572\1021585474.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
c:\Users\agarw\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


ModuleNotFoundError: No module named 'langchain_google_genai'

In [14]:
retriever.invoke("what is an agent?")

NameError: name 'retriever' is not defined

In [20]:
from langchain.chat_models import init_chat_model

In [21]:
llm = init_chat_model("gemini-3.1-flash-lite", model_provider="google_genai")
llm

ImportError: Initializing ChatGoogleGenerativeAI requires the langchain-google-genai package. Please install it with `pip install langchain-google-genai`

In [22]:
from langsmith import traceable

@traceable
def rag_bot(question: str) -> dict:
    docs = retriever.invoke(question)